In [0]:
spark.sql("""
ALTER TABLE workspace.eacc_ecommerce_rag.chunks
SET TBLPROPERTIES (
    delta.enableChangeDataFeed = true
)
""")

print("Change Data Feed enabled.")

In [0]:
spark.sql("""
SHOW TBLPROPERTIES workspace.eacc_ecommerce_rag.chunks
""").show(truncate=False)

In [0]:
%pip install -q databricks-ai-search
dbutils.library.restartPython()

In [0]:
from databricks.ai_search.client import AISearchClient

vsc = AISearchClient(
    disable_notice=True
)

print("AI Search client ready.")

In [0]:
endpoints = vsc.list_endpoints()

endpoints

In [0]:
AI_SEARCH_ENDPOINT = "eacc-rag-search"

In [0]:
endpoint = vsc.create_endpoint(
    name=AI_SEARCH_ENDPOINT,
    endpoint_type="STANDARD"
)

endpoint

In [0]:
endpoint_status = vsc.get_endpoint(
    name=AI_SEARCH_ENDPOINT
)

endpoint_status

In [0]:
SOURCE_TABLE = (
    "workspace.eacc_ecommerce_rag.chunks"
)

INDEX_NAME = (
    "workspace.eacc_ecommerce_rag."
    "enterprise_knowledge_index"
)

EMBEDDING_MODEL = (
    "databricks-bge-large-en"
)

In [0]:
index = vsc.create_delta_sync_index(
    endpoint_name=AI_SEARCH_ENDPOINT,

    source_table_name=SOURCE_TABLE,

    index_name=INDEX_NAME,

    pipeline_type="TRIGGERED",

    primary_key="chunk_id",

    embedding_source_column="chunk_text",

    embedding_model_endpoint_name=EMBEDDING_MODEL,

    columns_to_sync=[
        "document_id",
        "document_title",
        "document_owner",
        "document_type",
        "section_title",
        "section_order",
        "source_text",
        "chunk_character_count",
        "content_hash",
        "is_active"
    ]
)

index

In [0]:
import time

index = vsc.get_index(
    endpoint_name=AI_SEARCH_ENDPOINT,
    index_name=INDEX_NAME
)

while True:
    desc = index.describe()
    status = desc.get("status", {})
    
    print(
        "State:",
        status.get("detailed_state"),
        "| Ready:",
        status.get("ready"),
        "| Message:",
        status.get("message")
    )
    
    if status.get("ready") is True:
        print("Index is ready!")
        break
    
    time.sleep(10)

In [0]:
results = index.similarity_search(
    query_text=(
        "How long does a customer have "
        "to return a normal purchase?"
    ),

    columns=[
        "chunk_id",
        "document_title",
        "section_title",
        "source_text"
    ],

    num_results=5
)

results

In [0]:
results

In [0]:
def show_search_results(results):
    column_names = [
        column["name"]
        for column in results["manifest"]["columns"]
    ]

    rows = results["result"]["data_array"]

    for rank, row in enumerate(rows, start=1):
        # Search results normally include a score as the final value.
        values = row[:len(column_names)]
        item = dict(zip(column_names, values))

        score = row[-1] if len(row) > len(column_names) else None

        print("=" * 80)
        print("RANK:", rank)

        if score is not None:
            print("SCORE:", score)

        print("DOCUMENT:", item.get("document_title"))
        print("SECTION:", item.get("section_title"))
        print("CHUNK ID:", item.get("chunk_id"))
        print()
        print(item.get("source_text"))
        print()

In [0]:
show_search_results(results)

In [0]:
retrieval_tests = [
    "How long can a normal product be returned?",
    "How quickly does a damaged item need to be reported?",
    "At what category return rate is priority review required?",
    "When does a cancellation rate require priority review?",
    "What happens if a customer wants to cancel after shipment?",
    "What exactly does net revenue mean in our analytics model?"
]

In [0]:
for question in retrieval_tests:
    print("\n" + "#" * 100)
    print("QUESTION:", question)
    print()

    results = index.similarity_search(
        query_text=question,
        columns=[
            "chunk_id",
            "document_title",
            "section_title",
            "source_text"
        ],
        num_results=3
    )

    show_search_results(results)

In [0]:
semantic_test = index.similarity_search(
    query_text=(
        "I got the wrong thing in my package. "
        "Do I have to pay to send it back?"
    ),
    columns=[
        "chunk_id",
        "document_title",
        "section_title",
        "source_text"
    ],
    num_results=5
)

show_search_results(semantic_test)

In [0]:
ann_results = index.similarity_search(
    query_text="What does net revenue mean?",
    columns=[
        "document_title",
        "section_title",
        "source_text"
    ],
    num_results=5,
    query_type="ANN"
)

In [0]:
hybrid_results = index.similarity_search(
    query_text="What does net revenue mean?",
    columns=[
        "document_title",
        "section_title",
        "source_text"
    ],
    num_results=5,
    query_type="HYBRID"
)

In [0]:
filtered_results = index.similarity_search(
    query_text="How should damaged products be handled?",
    columns=[
        "document_title",
        "document_owner",
        "section_title",
        "source_text"
    ],
    filters={
        "document_owner": "Customer Operations"
    },
    num_results=5
)

show_search_results(filtered_results)

In [0]:
retrieval_eval = [
    {
        "question":
            "How long can a normal product be returned?",

        "acceptable_sections": [
            "Standard Return Window"
        ]
    },

    {
        "question":
            "How quickly must damaged merchandise be reported after delivery?",

        "acceptable_sections": [
            "Damaged or Defective Products",
            "Damaged or Defective Product Requests"
        ]
    },

    {
        "question":
            "At what monthly category return rate is priority review required?",

        "acceptable_sections": [
            "Category-Level Monitoring",
            "Category Return Rate",
            "Investigation Priority"
        ]
    },

    {
        "question":
            "When does the monthly cancellation rate trigger priority review?",

        "acceptable_sections": [
            "Cancellation Rate Monitoring",
            "Cancellation Rate"
        ]
    },

    {
        "question":
            "What happens if a customer wants to cancel an order after it has shipped?",

        "acceptable_sections": [
            "Shipped Orders",
            "Relationship to Shipping and Returns",
            "Cancellation Requests"
        ]
    },

    {
        "question":
            "What does net revenue mean in our analytics model?",

        "acceptable_sections": [
            "Net Revenue"
        ]
    },

    {
        "question":
            "Who pays return shipping when the customer receives the wrong item?",

        "acceptable_sections": [
            "Incorrect Items",
            "Incorrect Item Requests"
        ]
    },

    {
        "question":
            "How long should refund processing normally take after approval?",

        "acceptable_sections": [
            "Refund Processing",
            "Refund Questions"
        ]
    },

    {
        "question":
            "When can a shipment be considered potentially lost?",

        "acceptable_sections": [
            "Lost Shipments",
            "Delivery Delay Requests"
        ]
    },

    {
        "question":
            "What is the item reconciliation rate in the current dataset?",

        "acceptable_sections": [
            "Item Reconciliation Rate"
        ]
    }
]

In [0]:
def extract_search_rows(results):
    column_names = [
        column["name"]
        for column in results["manifest"]["columns"]
    ]

    rows = results["result"]["data_array"]

    parsed_rows = []

    for row in rows:
        values = row[:len(column_names)]

        parsed_rows.append(
            dict(zip(column_names, values))
        )

    return parsed_rows

In [0]:
test_results = index.similarity_search(
    query_text="What does net revenue mean?",
    columns=[
        "document_title",
        "section_title"
    ],
    num_results=3
)

extract_search_rows(test_results)

In [0]:
def evaluate_retrieval(query_type):
    hit_at_1 = 0
    hit_at_3 = 0
    reciprocal_rank_total = 0

    details = []

    for test in retrieval_eval:

        results = index.similarity_search(
            query_text=test["question"],
            columns=[
                "document_title",
                "section_title"
            ],
            num_results=3,
            query_type=query_type
        )

        rows = extract_search_rows(results)

        retrieved_sections = [
            row["section_title"]
            for row in rows
        ]

        acceptable = set(
            test["acceptable_sections"]
        )

        rank_found = None

        for rank, section in enumerate(
            retrieved_sections,
            start=1
        ):
            if section in acceptable:
                rank_found = rank
                break

        hit1 = rank_found == 1
        hit3 = rank_found is not None

        if hit1:
            hit_at_1 += 1

        if hit3:
            hit_at_3 += 1
            reciprocal_rank_total += (
                1 / rank_found
            )

        details.append({
            "question":
                test["question"],

            "expected":
                test["acceptable_sections"],

            "retrieved":
                retrieved_sections,

            "rank":
                rank_found,

            "hit@1":
                hit1,

            "hit@3":
                hit3
        })

    count = len(retrieval_eval)

    metrics = {
        "query_type": query_type,
        "hit_at_1": hit_at_1 / count,
        "hit_at_3": hit_at_3 / count,
        "mrr": reciprocal_rank_total / count
    }

    return metrics, details

In [0]:
ann_metrics, ann_details = (
    evaluate_retrieval("ann")
)

ann_metrics

In [0]:
for item in ann_details:
    print("=" * 100)

    print("QUESTION:")
    print(item["question"])

    print("\nEXPECTED:")
    print(item["expected"])

    print("\nRETRIEVED:")
    print(item["retrieved"])

    print("\nRANK:", item["rank"])
    print("HIT@1:", item["hit@1"])
    print("HIT@3:", item["hit@3"])

In [0]:
hybrid_metrics, hybrid_details = (
    evaluate_retrieval("hybrid")
)

hybrid_metrics

In [0]:
for item in hybrid_details:
    print("=" * 100)

    print("QUESTION:")
    print(item["question"])

    print("\nEXPECTED:")
    print(item["expected"])

    print("\nRETRIEVED:")
    print(item["retrieved"])

    print("\nRANK:", item["rank"])
    print("HIT@1:", item["hit@1"])
    print("HIT@3:", item["hit@3"])

In [0]:
print("ANN")
print(
    f"Hit@1: {ann_metrics['hit_at_1']:.2%}"
)
print(
    f"Hit@3: {ann_metrics['hit_at_3']:.2%}"
)
print(
    f"MRR:   {ann_metrics['mrr']:.3f}"
)

print()

print("HYBRID")
print(
    f"Hit@1: {hybrid_metrics['hit_at_1']:.2%}"
)
print(
    f"Hit@3: {hybrid_metrics['hit_at_3']:.2%}"
)
print(
    f"MRR:   {hybrid_metrics['mrr']:.3f}"
)

In [0]:
confusion_tests = [
    "How many days does a customer have for a normal return?",

    "How many days after delivery does a customer have to report a damaged product?",

    "What return rate causes standard review for an individual product?",

    "What return rate causes priority investigation for an individual product?",

    "What return rate causes standard review for a product category?",

    "What return rate causes priority review for a product category?",

    "At what cancellation rate does standard operational review begin?",

    "At what cancellation rate does priority review begin?"
]

In [0]:
BEST_QUERY_TYPE = (
    "hybrid"
    if hybrid_metrics["mrr"]
       >= ann_metrics["mrr"]
    else "ann"
)

print(
    "Selected retrieval strategy:",
    BEST_QUERY_TYPE
)

In [0]:
for question in confusion_tests:

    print("\n" + "=" * 100)
    print("QUESTION:", question)

    results = index.similarity_search(
        query_text=question,

        columns=[
            "document_title",
            "section_title",
            "source_text"
        ],

        num_results=3,

        query_type=BEST_QUERY_TYPE
    )

    show_search_results(results)

In [0]:
RETRIEVAL_QUERY_TYPE = BEST_QUERY_TYPE
RETRIEVAL_TOP_K = 3

print(
    "Query type:",
    RETRIEVAL_QUERY_TYPE
)

print(
    "Top K:",
    RETRIEVAL_TOP_K
)